# Kiến trúc light model (~1M tham số) cho CIFAR-10: MobileLite-C10

Mục tiêu là một mô hình nhỏ gọn, khoảng 1 triệu tham số, nhưng vẫn đủ mạnh để phân loại ảnh CIFAR-10 (32×32).  

## Ý tưởng chính
- Dùng **Depthwise Separable Convolution (DS-Conv)** để giảm tham số.  
- Khối cơ bản là **Inverted Residual Block** (còn gọi bottleneck).  
- Kích hoạt dùng **ReLU6** (giúp phân phối gọn hơn khi lượng tử).  
- Kết thúc bằng **Global Average Pooling (GAP)** thay vì fully-connected to.

## Cấu trúc MobileLite-C10
- **Stem**: Conv 3×3 (3→32α), stride=1, BN, ReLU6.  
- **Stage 1**: 1× bottleneck (t=4, k=3, stride=1, out=24).  
- **Stage 2**: 2× bottleneck (t=4, k=3, stride=2, out=32).  
- **Stage 3**: 2× bottleneck (t=4, k=3, stride=2, out=48).  
- **Stage 4**: 2× bottleneck (t=4, k=3, stride=1, out=64).  
- **Stage 5**: Conv 1×1 (64→128α), BN, ReLU6.  
- **Head**: GAP → FC (128α→10).  

Ở đây α là hệ số "width multiplier". Với α=0.75, mô hình khoảng 0.9–1.1M tham số.  

## Inverted Residual Block (bottleneck, t=4, k=3)
1. **Expand**: PW conv 1×1 mở rộng số kênh: $C_{mid} = t \cdot C_{in}$.  
2. **Depthwise conv**: DW conv 3×3 trên từng kênh (stride 1 hoặc 2).  
3. **Project**: PW conv 1×1 thu nhỏ về $C_{out}$.  
4. Nếu stride=1 và $C_{in}=C_{out}$ thì cộng residual.

Công thức block:

$$
Y =
\begin{cases}
X + \text{BN}(W_p * \phi(\text{BN}(W_d \odot \phi(\text{BN}(W_e * X))))) & \text{nếu stride=1, } C_{in}=C_{out} \\
\text{BN}(W_p * \phi(\text{BN}(W_d \odot \phi(\text{BN}(W_e * X))))) & \text{ngược lại}
\end{cases}
$$

Trong đó:
- $W_e$: kernel 1×1 (expand).  
- $W_d$: kernel depthwise 3×3.  
- $W_p$: kernel 1×1 (project).  
- $\phi$: ReLU6.  
- $\odot$: convolution từng kênh (depthwise).


In [1]:
# [Cell 1] Thiết lập môi trường & cấu hình chung
# - Import, seed, device, đường dẫn checkpoint, siêu tham số

from __future__ import annotations
import math, os, time, random
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.backends.cudnn.benchmark = True

def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METHOD_ABBR = "fp32"
MODEL_NAME = "MobileLiteC10"
CKPT_PATH = CHECKPOINT_DIR / f"{METHOD_ABBR}_{MODEL_NAME}.pth"

@dataclass
class CFG:
    epochs: int = 70
    batch_size: int = 128
    lr: float = 1e-3
    weight_decay: float = 5e-4
    num_workers: int = 2
    alpha: float = 2.0   # width multiplier

cfg = CFG()


In [2]:
# [Cell 2] Định nghĩa khối Inverted Residual và kiến trúc MobileLite-C10
# - Depthwise separable conv, ReLU6, GAP + FC

class ConvBNAct(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, g=1, act=True):
        super().__init__()
        p = (k - 1) // 2
        self.conv = nn.Conv2d(in_c, out_c, k, s, p, groups=g, bias=False)
        self.bn   = nn.BatchNorm2d(out_c)
        self.act  = act
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return F.relu6(x, inplace=True) if self.act else x

class InvertedResidual(nn.Module):
    def __init__(self, in_c, out_c, t=4, s=1, k=3):
        super().__init__()
        assert s in [1, 2]
        mid = int(round(in_c * t))
        self.use_res = (s == 1 and in_c == out_c)

        # 1x1 expand
        self.expand = ConvBNAct(in_c, mid, k=1, s=1, act=True) if t != 1 else nn.Identity()
        # 3x3 depthwise
        self.dw = ConvBNAct(mid, mid, k=k, s=s, g=mid, act=True)
        # 1x1 project (no activation)
        self.project = ConvBNAct(mid, out_c, k=1, s=1, act=False)

    def forward(self, x):
        y = self.expand(x) if not isinstance(self.expand, nn.Identity) else x
        y = self.dw(y)
        y = self.project(y)
        return x + y if self.use_res else y

class MobileLiteC10(nn.Module):
    def __init__(self, num_classes=10, alpha=0.75):
        super().__init__()
        def ch(x):  # channel helper with width multiplier
            return max(8, int(round(x * alpha)))

        # Stem
        self.stem = ConvBNAct(3, ch(32), k=3, s=1)

        # Stages (t=4, k=3 theo thiết kế)
        layers = []
        # Stage 1
        layers += [InvertedResidual(ch(32), ch(24), t=4, s=1, k=3),
                   InvertedResidual(ch(24), ch(24), t=4, s=1, k=3)]
        # Stage 2
        layers += [InvertedResidual(ch(24), ch(32), t=4, s=2, k=3),
                   InvertedResidual(ch(32), ch(32), t=4, s=1, k=3),
                   InvertedResidual(ch(32), ch(32), t=4, s=1, k=3)]
        # Stage 3
        layers += [InvertedResidual(ch(32), ch(48), t=4, s=2, k=3),
                   InvertedResidual(ch(48), ch(48), t=4, s=1, k=3),
                   InvertedResidual(ch(48), ch(48), t=4, s=1, k=3)]
        # Stage 4
        layers += [InvertedResidual(ch(48), ch(64), t=4, s=1, k=3),
                   InvertedResidual(ch(64), ch(64), t=4, s=1, k=3),
                   InvertedResidual(ch(64), ch(64), t=4, s=1, k=3)]
        self.features = nn.Sequential(*layers)

        # Conv 1x1 cuối + Head
        self.head_conv = ConvBNAct(ch(64), ch(160), k=1, s=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(ch(160), num_classes)

        # Khởi tạo chuẩn
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = self.head_conv(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [3]:
# [Cell 3] Dataloader CIFAR-10 (train/val/test) với augmentation nhẹ
# - Chuẩn hóa theo mean/std CIFAR-10, chia train/val

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

root = "./data"
full_train = datasets.CIFAR10(root, train=True, transform=train_tf, download=True)
test_set   = datasets.CIFAR10(root, train=False, transform=test_tf, download=True)

# Tách train/val
val_ratio = 0.1
val_size = int(len(full_train) * val_ratio)
train_size = len(full_train) - val_size
train_set, val_set = torch.utils.data.random_split(full_train, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)


100%|██████████| 170M/170M [00:05<00:00, 29.6MB/s] 


In [4]:
# [Cell 4] Vòng lặp train/eval gọn gàng
# - CrossEntropyLoss, AdamW, AMP
# - Ghi loss/accuracy mỗi epoch; lưu best checkpoint theo validation

from torch.cuda.amp import autocast, GradScaler

def accuracy(output: torch.Tensor, target: torch.Tensor, topk=(1,)):
    maxk = max(topk)
    with torch.no_grad():
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1))
        res = []
        for k in topk:
            res.append(correct[:k].reshape(-1).float().sum().item() / target.size(0))
        return res  # tỷ lệ [0..1]

def run_epoch(model, loader, optimizer=None, scaler: GradScaler | None = None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, total_correct, total_n = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()

    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=device.type=="cuda"):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.inference_mode():
                logits = model(x)
                loss = criterion(logits, y)

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_n += bs
        total_correct += (logits.argmax(dim=1) == y).sum().item()

    avg_loss = total_loss / total_n
    top1 = total_correct / total_n
    return avg_loss, top1

def train_main():
    model = MobileLiteC10(num_classes=10, alpha=cfg.alpha).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = GradScaler(enabled=device.type=="cuda")

    best_val_top1 = -1.0
    start = time.time()
    for epoch in range(1, cfg.epochs + 1):
        tr_loss, tr_top1 = run_epoch(model, train_loader, optimizer, scaler)
        va_loss, va_top1 = run_epoch(model, val_loader, optimizer=None, scaler=None)

        # Lưu best theo validation
        if va_top1 > best_val_top1:
            best_val_top1 = va_top1
            torch.save({"model": model.state_dict(),
                        "epoch": epoch,
                        "val_top1": best_val_top1,
                        "cfg": cfg.__dict__}, CKPT_PATH)

        # Log gọn mỗi epoch
        print(f"Epoch {epoch:02d}/{cfg.epochs} | "
              f"train CE: {tr_loss:.4f} acc@1: {tr_top1*100:.2f}% | "
              f"val CE: {va_loss:.4f} acc@1: {va_top1*100:.2f}%")

    dur = time.time() - start
    print(f"Hoàn tất huấn luyện trong {dur/60:.1f} phút. Best val acc@1: {best_val_top1*100:.2f}%")
    # In số tham số để tiện theo dõi ngay sau train
    print(f"Params: {count_params(model):,}")

# Bắt đầu huấn luyện
train_main()


/tmp/ipykernel_36/1694673321.py:52: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=device.type=="cuda")
/tmp/ipykernel_36/1694673321.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=device.type=="cuda"):


Epoch 01/70 | train CE: 1.4135 acc@1: 47.15% | val CE: 1.1596 acc@1: 58.12%
Epoch 02/70 | train CE: 0.9466 acc@1: 66.32% | val CE: 0.9189 acc@1: 67.04%
Epoch 03/70 | train CE: 0.7468 acc@1: 73.32% | val CE: 0.6772 acc@1: 76.08%
Epoch 04/70 | train CE: 0.6278 acc@1: 78.06% | val CE: 0.6359 acc@1: 77.70%
Epoch 05/70 | train CE: 0.5444 acc@1: 81.13% | val CE: 0.5609 acc@1: 80.80%
Epoch 06/70 | train CE: 0.4849 acc@1: 83.25% | val CE: 0.5170 acc@1: 82.42%
Epoch 07/70 | train CE: 0.4466 acc@1: 84.49% | val CE: 0.5233 acc@1: 81.64%
Epoch 08/70 | train CE: 0.4119 acc@1: 85.77% | val CE: 0.4927 acc@1: 82.64%
Epoch 09/70 | train CE: 0.3777 acc@1: 86.87% | val CE: 0.4744 acc@1: 83.28%
Epoch 10/70 | train CE: 0.3498 acc@1: 87.87% | val CE: 0.4609 acc@1: 84.82%
Epoch 11/70 | train CE: 0.3370 acc@1: 88.33% | val CE: 0.3972 acc@1: 86.24%
Epoch 12/70 | train CE: 0.3148 acc@1: 89.07% | val CE: 0.3922 acc@1: 86.44%
Epoch 13/70 | train CE: 0.2984 acc@1: 89.61% | val CE: 0.3951 acc@1: 86.34%
Epoch 14/70 

In [5]:
# [Cell 5] Kiểm tra nhanh checkpoint đã ghi (tùy chọn, có thể bỏ qua khi chạy lại notebook)
# - Xác nhận file tồn tại

assert CKPT_PATH.exists(), f"Checkpoint chưa được tạo tại {CKPT_PATH}"
print(f"Checkpoint đã lưu: {CKPT_PATH}")


Checkpoint đã lưu: checkpoints/fp32_MobileLiteC10.pth


In [8]:
# [Cell 6] Tiện ích đo lường: FLOPs/MACs, latency, top-k accuracy
# - Hàm count_macs_flops bằng forward-hook (Conv2d, Linear)
# - Hàm eval_model: tính top-1/top-5, thời gian infer/batch & per-sample

from contextlib import contextmanager

def count_macs_flops(model: nn.Module, input_size=(1, 3, 32, 32)):
    hooks = []
    macs = 0

    def conv_hook(m, inp, out):
        nonlocal macs
        # inp[0]: [B, Cin, Hin, Win], out: [B, Cout, Hout, Wout]
        x = inp[0]
        B, Cin, Hin, Win = x.shape
        B, Cout, Hout, Wout = out.shape
        Kh, Kw = m.kernel_size
        groups = m.groups
        cin_per_group = Cin // groups
        # MACs cho Conv2d
        macs_layer = Cout * Hout * Wout * Kh * Kw * cin_per_group
        macs += macs_layer

    def linear_hook(m, inp, out):
        nonlocal macs
        # MACs cho Linear
        in_f = inp[0].shape[-1]
        out_f = out.shape[-1]
        macs += in_f * out_f

    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))

    model.eval()
    dummy = torch.randn(*input_size, device=next(model.parameters()).device)
    with torch.inference_mode():
        _ = model(dummy)

    for h in hooks:
        h.remove()

    flops = 2 * macs  # 1 MAC ~ 2 FLOPs (mul + add)
    return macs, flops

@contextmanager
def cuda_sync():
    try:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        yield
    finally:
        if torch.cuda.is_available():
            torch.cuda.synchronize()

def eval_model(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    top1_correct = 0
    top5_correct = 0
    total = 0
    t0 = time.time()
    infer_times = []

    with torch.inference_mode(), cuda_sync():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            # warmup nhẹ (chỉ lần đầu trên CUDA)
            if not infer_times and device.type == "cuda":
                _ = model(x)

            t_batch0 = time.time()
            logits = model(x)
            with cuda_sync():
                pass
            t_batch1 = time.time()
            infer_times.append(t_batch1 - t_batch0)

            # top-1/top-5
            maxk = 5
            _, pred = logits.topk(maxk, 1, True, True)
            pred = pred.t()
            correct = pred.eq(y.view(1, -1))
            top1_correct += correct[:1].reshape(-1).float().sum().item()
            top5_correct += correct[:5].reshape(-1).float().sum().item()
            total += y.size(0)

    elapsed = time.time() - t0
    top1 = top1_correct / total
    top5 = top5_correct / total
    avg_batch_time = sum(infer_times) / len(infer_times)
    avg_per_sample_time = avg_batch_time / loader.batch_size
    return {
        "top1": top1,
        "top5": top5,
        "elapsed_s": elapsed,
        "avg_batch_time_s": avg_batch_time,
        "avg_per_sample_time_s": avg_per_sample_time,
    }


In [10]:
# [Cell 7] ĐÁNH GIÁ ĐỘC LẬP TỪ CHECKPOINT (final eval cell cho phần FP32 baseline)
# - Không phụ thuộc biến trước đó: định nghĩa kiến trúc, dataloader test, load ckpt và đo metrics

import os, time, math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ==== Cấu hình ====
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = Path("checkpoints")
METHOD_ABBR = "fp32"
MODEL_NAME = "MobileLiteC10"
CKPT_PATH = CHECKPOINT_DIR / f"{METHOD_ABBR}_{MODEL_NAME}.pth"

# ==== Định nghĩa kiến trúc ====
class ConvBNAct(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, g=1, act=True):
        super().__init__()
        p = (k - 1) // 2
        self.conv = nn.Conv2d(in_c, out_c, k, s, p, groups=g, bias=False)
        self.bn   = nn.BatchNorm2d(out_c)
        self.act  = act
    def forward(self, x):
        x = self.conv(x); x = self.bn(x)
        return F.relu6(x, inplace=True) if self.act else x

class InvertedResidual(nn.Module):
    def __init__(self, in_c, out_c, t=4, s=1, k=3):
        super().__init__()
        assert s in [1, 2]
        mid = int(round(in_c * t))
        self.use_res = (s == 1 and in_c == out_c)
        self.expand = ConvBNAct(in_c, mid, k=1, s=1, act=True) if t != 1 else nn.Identity()
        self.dw = ConvBNAct(mid, mid, k=k, s=s, g=mid, act=True)
        self.project = ConvBNAct(mid, out_c, k=1, s=1, act=False)
    def forward(self, x):
        y = self.expand(x) if not isinstance(self.expand, nn.Identity) else x
        y = self.dw(y); y = self.project(y)
        return x + y if self.use_res else y

class MobileLiteC10(nn.Module):
    def __init__(self, num_classes=10, alpha=2.0):
        super().__init__()
        def ch(x): return max(8, int(round(x * alpha)))
        self.stem = ConvBNAct(3, ch(32), k=3, s=1)
        layers = []
        # Stage 1
        layers += [InvertedResidual(ch(32), ch(24), t=4, s=1, k=3),
                   InvertedResidual(ch(24), ch(24), t=4, s=1, k=3)]
        # Stage 2
        layers += [InvertedResidual(ch(24), ch(32), t=4, s=2, k=3),
                   InvertedResidual(ch(32), ch(32), t=4, s=1, k=3),
                   InvertedResidual(ch(32), ch(32), t=4, s=1, k=3)]
        # Stage 3
        layers += [InvertedResidual(ch(32), ch(48), t=4, s=2, k=3),
                   InvertedResidual(ch(48), ch(48), t=4, s=1, k=3),
                   InvertedResidual(ch(48), ch(48), t=4, s=1, k=3)]
        # Stage 4
        layers += [InvertedResidual(ch(48), ch(64), t=4, s=1, k=3),
                   InvertedResidual(ch(64), ch(64), t=4, s=1, k=3),
                   InvertedResidual(ch(64), ch(64), t=4, s=1, k=3)]
        self.features = nn.Sequential(*layers)
        self.head_conv = ConvBNAct(ch(64), ch(160), k=1, s=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(ch(160), num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = self.head_conv(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ==== DataLoader test ====
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)])
test_set = datasets.CIFAR10("./data", train=False, transform=test_tf, download=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

# ==== Load checkpoint ====
assert CKPT_PATH.exists(), f"Không tìm thấy checkpoint tại {CKPT_PATH}"
ckpt = torch.load(CKPT_PATH, map_location="cpu")
alpha = ckpt.get("cfg", {}).get("alpha", 0.75)
model = MobileLiteC10(num_classes=10, alpha=alpha).to(DEVICE)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()

# ==== Đo tham số, MACs/FLOPs, latency và accuracy ====
def count_macs_flops(model: nn.Module, input_size=(1, 3, 32, 32)):
    hooks = []
    macs = 0
    def conv_hook(m, inp, out):
        nonlocal macs
        x = inp[0]
        B, Cin, Hin, Win = x.shape
        B, Cout, Hout, Wout = out.shape
        Kh, Kw = m.kernel_size
        groups = m.groups
        cin_per_group = Cin // groups
        macs += Cout * Hout * Wout * Kh * Kw * cin_per_group
    def linear_hook(m, inp, out):
        nonlocal macs
        in_f = inp[0].shape[-1]; out_f = out.shape[-1]
        macs += in_f * out_f
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))
    with torch.inference_mode():
        dummy = torch.randn(*input_size, device=DEVICE)
        _ = model(dummy)
    for h in hooks: h.remove()
    flops = 2 * macs
    return macs, flops

@torch.inference_mode()
def evaluate(model, loader):
    top1 = 0; top5 = 0; total = 0
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    times = []
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        if not times and DEVICE.type == "cuda":
            _ = model(x)  # warmup
            torch.cuda.synchronize()
        t_b0 = time.time()
        logits = model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t_b1 = time.time()
        times.append(t_b1 - t_b0)

        _, pred = logits.topk(5, 1, True, True)
        pred = pred.t()
        correct = pred.eq(y.view(1, -1))
        top1 += correct[:1].reshape(-1).float().sum().item()
        top5 += correct[:5].reshape(-1).float().sum().item()
        total += y.size(0)
    elapsed = time.time() - t0
    return {
        "top1": top1 / total,
        "top5": top5 / total,
        "elapsed_s": elapsed,
        "avg_batch_time_s": sum(times)/len(times),
        "avg_per_sample_time_s": (sum(times)/len(times)) / loader.batch_size
    }

params = count_params(model)
macs, flops = count_macs_flops(model, input_size=(1,3,32,32))
metrics = evaluate(model, test_loader)

print(f"Params: {params:,}")
print(f"MACs (32x32): {macs/1e6:.2f} M, FLOPs: {flops/1e6:.2f} MFLOPs")
print(f"Test acc@1: {metrics['top1']*100:.2f}% | acc@5: {metrics['top5']*100:.2f}%")
print(f"Total eval time: {metrics['elapsed_s']:.2f}s")
print(f"Avg batch latency: {metrics['avg_batch_time_s']*1000:.3f} ms")
print(f"Avg per-sample latency: {metrics['avg_per_sample_time_s']*1000:.3f} ms")


Params: 765,898
MACs (32x32): 126.63 M, FLOPs: 253.25 MFLOPs
Test acc@1: 90.34% | acc@5: 99.73%
Total eval time: 2.09s
Avg batch latency: 21.998 ms
Avg per-sample latency: 0.172 ms


# PTQ Phiên bản 1 — MinMax Calibration (chuẩn cơ bản)

## Ý tưởng
- Weights lượng tử **per-channel symmetric** (mỗi kênh có thang đo riêng, zero-point = 0).  
- Activations lượng tử **per-tensor asymmetric** (toàn tensor chung scale, có zero-point).  
- Calibration bằng min/max đơn giản, nhanh, dùng khoảng 512–1024 ảnh.

## Công thức lượng tử
- Quantize:  
  $$
  q = \text{clip}\left(\left\lfloor \frac{x}{s} \right\rceil + z,\ q_{min}, q_{max}\right)
  $$
- Dequantize:  
  $$
  \hat{x} = s \cdot (q - z)
  $$
- Với activations (asymmetric):  
  $$
  s = \frac{x_{max} - x_{min}}{q_{max} - q_{min}}, \quad
  z = \left\lfloor q_{min} - \frac{x_{min}}{s} \right\rceil
  $$
- Với weights (symmetric, per-channel):  
  $$
  s_c = \frac{\max(|x_{min,c}|, |x_{max,c}|)}{127}, \quad z_c = 0
  $$

## Trình tự
1. Train mô hình FP32 → lưu checkpoint.  
2. Fuse Conv–BN–ReLU.  
3. Gắn observer (weights per-channel, activations per-tensor).  
4. Chạy calibration (512–1024 ảnh không augment).  
5. Convert sang INT8.  
6. Đánh giá: accuracy, size, latency.  
7. Nếu accuracy drop nhiều, tăng ảnh calibration hoặc dùng clipping 99.9%.


In [13]:
# [Cell 8] PTQ v1 (static, MinMax) — Chuẩn bị & tiện ích
# - Load FP32 baseline, chọn backend, prepare/calibrate/convert, lưu checkpoint INT8
# - Có hàm đo kích thước file, hiệu năng, và so sánh với baseline

from __future__ import annotations
import os, time, math
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from torch.ao.quantization import get_default_qconfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

# ==== Định nghĩa lại kiến trúc (tận dụng từ phần train) ====
class ConvBNAct(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, g=1, act=True):
        super().__init__()
        p = (k - 1) // 2
        self.conv = nn.Conv2d(in_c, out_c, k, s, p, groups=g, bias=False)
        self.bn   = nn.BatchNorm2d(out_c)
        self.act  = act
    def forward(self, x):
        x = self.conv(x); x = self.bn(x)
        return F.relu6(x, inplace=True) if self.act else x

class InvertedResidual(nn.Module):
    def __init__(self, in_c, out_c, t=4, s=1, k=3):
        super().__init__()
        assert s in [1, 2]
        mid = int(round(in_c * t))
        self.use_res = (s == 1 and in_c == out_c)
        self.expand = ConvBNAct(in_c, mid, k=1, s=1, act=True) if t != 1 else nn.Identity()
        self.dw = ConvBNAct(mid, mid, k=k, s=s, g=mid, act=True)
        self.project = ConvBNAct(mid, out_c, k=1, s=1, act=False)
    def forward(self, x):
        y = self.expand(x) if not isinstance(self.expand, nn.Identity) else x
        y = self.dw(y); y = self.project(y)
        return x + y if self.use_res else y

class MobileLiteC10(nn.Module):
    def __init__(self, num_classes=10, alpha=0.75):
        super().__init__()
        def ch(x): return max(8, int(round(x * alpha)))
        self.stem = ConvBNAct(3, ch(32), k=3, s=1)
        layers = []
        # Giữ cấu hình đã tăng block/alpha theo phần bạn đã chỉnh
        layers += [InvertedResidual(ch(32), ch(24), t=4, s=1, k=3),
                   InvertedResidual(ch(24), ch(24), t=4, s=1, k=3)]
        layers += [InvertedResidual(ch(24), ch(32), t=4, s=2, k=3),
                   InvertedResidual(ch(32), ch(32), t=4, s=1, k=3),
                   InvertedResidual(ch(32), ch(32), t=4, s=1, k=3)]
        layers += [InvertedResidual(ch(32), ch(48), t=4, s=2, k=3),
                   InvertedResidual(ch(48), ch(48), t=4, s=1, k=3),
                   InvertedResidual(ch(48), ch(48), t=4, s=1, k=3)]
        layers += [InvertedResidual(ch(48), ch(64), t=4, s=1, k=3),
                   InvertedResidual(ch(64), ch(64), t=4, s=1, k=3),
                   InvertedResidual(ch(64), ch(64), t=4, s=1, k=3)]
        self.features = nn.Sequential(*layers)
        self.head_conv = ConvBNAct(ch(64), ch(160), k=1, s=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(ch(160), num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = self.head_conv(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_macs_flops(model: nn.Module, input_size=(1,3,32,32)):
    hooks = []
    macs = 0
    def conv_hook(m, inp, out):
        nonlocal macs
        x = inp[0]
        B, Cin, Hin, Win = x.shape
        B, Cout, Hout, Wout = out.shape
        Kh, Kw = m.kernel_size
        groups = m.groups
        cin_per_group = Cin // groups
        macs += Cout * Hout * Wout * Kh * Kw * cin_per_group
    def linear_hook(m, inp, out):
        nonlocal macs
        in_f = inp[0].shape[-1]; out_f = out.shape[-1]
        macs += in_f * out_f
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))
    model.eval()
    with torch.inference_mode():
        dummy = torch.randn(*input_size)
        _ = model(dummy)
    for h in hooks: h.remove()
    flops = 2 * macs
    return macs, flops

def eval_model_cpu(model: nn.Module, loader: DataLoader):
    model.eval()
    top1 = top5 = total = 0
    t_list = []
    with torch.inference_mode():
        for x, y in loader:
            # warmup một batch đầu
            if not t_list:
                _ = model(x)
            t0 = time.time()
            logits = model(x)
            t1 = time.time()
            t_list.append(t1 - t0)

            _, pred = logits.topk(5, 1, True, True)
            pred = pred.t()
            correct = pred.eq(y.view(1, -1))
            top1 += correct[:1].reshape(-1).float().sum().item()
            top5 += correct[:5].reshape(-1).float().sum().item()
            total += y.size(0)
    avg_batch = sum(t_list)/len(t_list)
    return {
        "top1": top1/total, "top5": top5/total,
        "avg_batch_time_s": avg_batch,
        "avg_per_sample_time_s": avg_batch/loader.batch_size
    }

def file_size_mb(path: Path) -> float:
    return Path(path).stat().st_size / (1024*1024)

# ==== Cấu hình/đường dẫn ====
CHECKPOINT_DIR = Path("checkpoints")
FP32_CKPT = CHECKPOINT_DIR / "fp32_MobileLiteC10.pth"
PTQ_CKPT  = CHECKPOINT_DIR / "ptqv1_MobileLiteC10.pth"

# ==== Data cho calibration và test (không augment cho calib/test) ====
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)
calib_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)])
test_tf  = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)])

calib_full = datasets.CIFAR10("./data", train=True,  transform=calib_tf, download=True)
test_set   = datasets.CIFAR10("./data", train=False, transform=test_tf,   download=True)

# chọn 1024 ảnh calibration đại diện
idxs = list(range(min(1024, len(calib_full))))
calib_set = Subset(calib_full, idxs)
calib_loader = DataLoader(calib_set, batch_size=128, shuffle=False, num_workers=2, pin_memory=False)
test_loader_cpu = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2, pin_memory=False)

# ==== Backend lựa chọn tự động ====
supported = torch.backends.quantized.supported_engines
engine = "fbgemm" if "fbgemm" in supported else ("qnnpack" if "qnnpack" in supported else supported[0])
torch.backends.quantized.engine = engine


In [15]:
# [Cell 9] PTQ v1 (static, MinMax) — Chuẩn hoá/chuẩn bị → Calibration → Convert → Lưu checkpoint → So sánh baseline vs INT8

import torch
from torch.ao.quantization import get_default_qconfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

# 1) Load baseline FP32
assert FP32_CKPT.exists(), f"Không tìm thấy baseline tại {FP32_CKPT}"
ckpt = torch.load(FP32_CKPT, map_location="cpu")
alpha = ckpt.get("cfg", {}).get("alpha", 0.75)
fp32_model = MobileLiteC10(num_classes=10, alpha=alpha).eval()
fp32_model.load_state_dict(ckpt["model"], strict=True)

# 2) Đo baseline (CPU) cho so sánh công bằng latency
macs, flops = count_macs_flops(fp32_model, input_size=(1,3,32,32))
base_params = count_params(fp32_model)
base_metrics = eval_model_cpu(fp32_model, test_loader_cpu)
base_ckpt_mb = file_size_mb(FP32_CKPT)

# 3) Chuẩn bị PTQ: qconfig per-tensor act, per-channel sym weight (mặc định backend)
qconfig = get_default_qconfig(engine)
qconfig_dict = {"": qconfig}
example_input = torch.randn(1,3,32,32)
prepared = prepare_fx(fp32_model, qconfig_dict, example_input)

# 4) Calibration (MinMax observers)
with torch.inference_mode():
    for x, _ in calib_loader:
        _ = prepared(x)

# 5) Convert → INT8
int8_model = convert_fx(prepared).eval()

# 6) Lưu checkpoint INT8 (state_dict + meta)
torch.save({
    "state_dict": int8_model.state_dict(),
    "alpha": alpha,
    "engine": engine,
    "example_input_shape": (1,3,32,32)
}, PTQ_CKPT)

# 7) Đánh giá INT8 trên CPU
int8_metrics = eval_model_cpu(int8_model, test_loader_cpu)
ptq_ckpt_mb = file_size_mb(PTQ_CKPT)

# 8) In kết quả so sánh
print("=== Baseline FP32 (CPU) ===")
print(f"Params: {base_params:,}")
print(f"MACs: {macs/1e6:.2f} M | FLOPs: {flops/1e6:.2f} MFLOPs (hình học như nhau cho FP32 & INT8)")
print(f"Checkpoint size: {base_ckpt_mb:.2f} MB")
print(f"acc@1: {base_metrics['top1']*100:.2f}% | acc@5: {base_metrics['top5']*100:.2f}%")
print(f"Avg batch latency: {base_metrics['avg_batch_time_s']*1000:.3f} ms | "
      f"per-sample: {base_metrics['avg_per_sample_time_s']*1000:.3f} ms")

print("\n=== PTQ v1 INT8 (CPU) ===")
print(f"Params (số lượng logic tương đương, lưu ý int8 không còn là nn.Parameter toàn bộ).")
print(f"MACs: {macs/1e6:.2f} M | FLOPs: {flops/1e6:.2f} MFLOPs")
print(f"Checkpoint size: {ptq_ckpt_mb:.2f} MB")
print(f"acc@1: {int8_metrics['top1']*100:.2f}% | acc@5: {int8_metrics['top5']*100:.2f}%")
print(f"Avg batch latency: {int8_metrics['avg_batch_time_s']*1000:.3f} ms | "
      f"per-sample: {int8_metrics['avg_per_sample_time_s']*1000:.3f} ms")

improve_ms = (base_metrics['avg_per_sample_time_s'] - int8_metrics['avg_per_sample_time_s']) * 1000
print(f"\nLatency per-sample Δ (FP32 → INT8): {improve_ms:+.3f} ms "
      f"(>0 nghĩa là INT8 nhanh hơn).")
print(f"Checkpoint Δ size: {base_ckpt_mb - ptq_ckpt_mb:+.2f} MB")


/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/quantize_fx.py:146: FutureWarning: Passing a QConfig dictionary to prepare is deprecated and will not be supported in a future version. Please pass in a QConfigMapping instead.
  prepared = prepare(
/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


=== Baseline FP32 (CPU) ===
Params: 765,898
MACs: 126.63 M | FLOPs: 253.25 MFLOPs (hình học như nhau cho FP32 & INT8)
Checkpoint size: 3.06 MB
acc@1: 90.34% | acc@5: 99.73%
Avg batch latency: 2215.401 ms | per-sample: 17.308 ms

=== PTQ v1 INT8 (CPU) ===
Params (số lượng logic tương đương, lưu ý int8 không còn là nn.Parameter toàn bộ).
MACs: 126.63 M | FLOPs: 253.25 MFLOPs
Checkpoint size: 0.95 MB
acc@1: 90.16% | acc@5: 99.73%
Avg batch latency: 420.958 ms | per-sample: 3.289 ms

Latency per-sample Δ (FP32 → INT8): +14.019 ms (>0 nghĩa là INT8 nhanh hơn).
Checkpoint Δ size: +2.11 MB


In [16]:
# [Cell 10] PTQ v1 — FINAL EVAL CELL (độc lập): load checkpoint INT8 và đánh giá đầy đủ
# - Cell này tự định nghĩa model, chuẩn hoá quantization graph, load state_dict INT8 và đo metrics
# - Hiển thị: params (logic), FLOPs, latency, top-1/top-5, thời gian infer trung bình/mẫu

import os, time
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.ao.quantization import get_default_qconfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

# ==== Kiến trúc ====
class ConvBNAct(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, g=1, act=True):
        super().__init__()
        p = (k - 1) // 2
        self.conv = nn.Conv2d(in_c, out_c, k, s, p, groups=g, bias=False)
        self.bn   = nn.BatchNorm2d(out_c)
        self.act  = act
    def forward(self, x):
        x = self.conv(x); x = self.bn(x)
        return F.relu6(x, inplace=True) if self.act else x

class InvertedResidual(nn.Module):
    def __init__(self, in_c, out_c, t=4, s=1, k=3):
        super().__init__()
        assert s in [1, 2]
        mid = int(round(in_c * t))
        self.use_res = (s == 1 and in_c == out_c)
        self.expand = ConvBNAct(in_c, mid, k=1, s=1, act=True) if t != 1 else nn.Identity()
        self.dw = ConvBNAct(mid, mid, k=k, s=s, g=mid, act=True)
        self.project = ConvBNAct(mid, out_c, k=1, s=1, act=False)
    def forward(self, x):
        y = self.expand(x) if not isinstance(self.expand, nn.Identity) else x
        y = self.dw(y); y = self.project(y)
        return x + y if self.use_res else y

class MobileLiteC10(nn.Module):
    def __init__(self, num_classes=10, alpha=0.75):
        super().__init__()
        def ch(x): return max(8, int(round(x * alpha)))
        self.stem = ConvBNAct(3, ch(32), k=3, s=1)
        layers = []
        layers += [InvertedResidual(ch(32), ch(24), t=4, s=1, k=3),
                   InvertedResidual(ch(24), ch(24), t=4, s=1, k=3)]
        layers += [InvertedResidual(ch(24), ch(32), t=4, s=2, k=3),
                   InvertedResidual(ch(32), ch(32), t=4, s=1, k=3),
                   InvertedResidual(ch(32), ch(32), t=4, s=1, k=3)]
        layers += [InvertedResidual(ch(32), ch(48), t=4, s=2, k=3),
                   InvertedResidual(ch(48), ch(48), t=4, s=1, k=3),
                   InvertedResidual(ch(48), ch(48), t=4, s=1, k=3)]
        layers += [InvertedResidual(ch(48), ch(64), t=4, s=1, k=3),
                   InvertedResidual(ch(64), ch(64), t=4, s=1, k=3),
                   InvertedResidual(ch(64), ch(64), t=4, s=1, k=3)]
        self.features = nn.Sequential(*layers)
        self.head_conv = ConvBNAct(ch(64), ch(160), k=1, s=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(ch(160), num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = self.head_conv(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Hỗ trợ đếm MACs cho cả Conv2d thường và Conv2d lượng tử
def count_macs_flops_any(model: nn.Module, input_size=(1,3,32,32)):
    import torch.nn.quantized as nnq
    hooks = []
    macs = 0
    def conv_hook(m, inp, out):
        nonlocal macs
        x = inp[0]
        B, Cin, Hin, Win = x.shape
        B, Cout, Hout, Wout = out.shape
        Kh, Kw = m.kernel_size
        groups = getattr(m, "groups", 1)
        cin_per_group = Cin // groups
        macs += Cout * Hout * Wout * Kh * Kw * cin_per_group
    def linear_hook(m, inp, out):
        nonlocal macs
        in_f = inp[0].shape[-1]; out_f = out.shape[-1]
        macs += in_f * out_f
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nnq.Conv2d)):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, (nn.Linear, nnq.Linear)):
            hooks.append(m.register_forward_hook(linear_hook))
    model.eval()
    with torch.inference_mode():
        dummy = torch.randn(*input_size)
        _ = model(dummy)
    for h in hooks: h.remove()
    flops = 2 * macs
    return macs, flops

@torch.inference_mode()
def evaluate_cpu(model: nn.Module, loader: DataLoader):
    model.eval()
    top1 = top5 = total = 0
    times = []
    for x, y in loader:
        if not times:
            _ = model(x)  # warmup
        t0 = time.time()
        logits = model(x)
        t1 = time.time()
        times.append(t1 - t0)
        _, pred = logits.topk(5, 1, True, True)
        pred = pred.t()
        correct = pred.eq(y.view(1, -1))
        top1 += correct[:1].reshape(-1).float().sum().item()
        top5 += correct[:5].reshape(-1).float().sum().item()
        total += y.size(0)
    avg_batch = sum(times)/len(times)
    return {
        "top1": top1/total, "top5": top5/total,
        "avg_batch_time_s": avg_batch,
        "avg_per_sample_time_s": avg_batch/loader.batch_size
    }

# ==== Load checkpoint INT8 ====
CKPT = Path("checkpoints/ptqv1_MobileLiteC10.pth")
assert CKPT.exists(), f"Không tìm thấy checkpoint: {CKPT}"
blob = torch.load(CKPT, map_location="cpu")
alpha = blob.get("alpha", 0.75)
engine = blob.get("engine", "fbgemm")
example_shape = blob.get("example_input_shape", (1,3,32,32))
torch.backends.quantized.engine = engine

# ==== Xây graph lượng tử & load state_dict ====
fp32_model = MobileLiteC10(alpha=alpha).eval()
qconfig = get_default_qconfig(engine)
qconfig_dict = {"": qconfig}
example_input = torch.randn(*example_shape)
prepared = prepare_fx(fp32_model, qconfig_dict, example_input)
int8_model = convert_fx(prepared).eval()
int8_model.load_state_dict(blob["state_dict"], strict=True)

# ==== DataLoader test (CPU) ====
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)])
test_set = datasets.CIFAR10("./data", train=False, transform=test_tf, download=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2, pin_memory=False)

# ==== Đo metrics ====
params_logic = count_params(fp32_model)  # số tham số logic (trước lượng tử, FLOPs như nhau)
macs, flops = count_macs_flops_any(int8_model, input_size=example_shape)
metrics = evaluate_cpu(int8_model, test_loader)

print(f"Engine: {engine}")
print(f"Params (logic FP32 graph): {params_logic:,}")
print(f"MACs: {macs/1e6:.2f} M | FLOPs: {flops/1e6:.2f} MFLOPs")
print(f"acc@1: {metrics['top1']*100:.2f}% | acc@5: {metrics['top5']*100:.2f}%")
print(f"Avg batch latency: {metrics['avg_batch_time_s']*1000:.3f} ms | "
      f"Avg per-sample: {metrics['avg_per_sample_time_s']*1000:.3f} ms")


/usr/local/lib/python3.11/dist-packages/torch/_utils.py:431: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,
/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/quantize_fx.py:146: FutureWarning: Passing a QConfig dictionary to prepare is deprecated and will not be supported in a future version. Please pass in a QConfigMapping instead.
  prepared = prepare(
/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/observer.py:1318: UserWarning: must run ob

Engine: fbgemm
Params (logic FP32 graph): 765,898
MACs: 126.63 M | FLOPs: 253.25 MFLOPs
acc@1: 90.16% | acc@5: 99.73%
Avg batch latency: 439.778 ms | Avg per-sample: 3.436 ms


# PTQ Phiên bản 2 — Histogram/KL Calibration + Bias Correction

## Ý tưởng
- MinMax dễ bị outlier. Thay vào đó, dùng histogram và chọn ngưỡng tối ưu bằng KL-divergence (hoặc MSE).  
- Sau lượng tử, áp dụng **bias correction** để bù lại sai số trung bình.

## Calibration bằng histogram + KL
- Xây histogram $H$ của activation.  
- Tìm ngưỡng $T$ để phân bố lượng tử $\tilde{P}_T$ gần với phân bố gốc $P$:  
  $$
  T^\star = \arg\min_T \mathrm{KL}(P(\cdot \mid |x| \le T)\,\|\,\tilde{P}_T)
  $$
- Sau đó scale: $s = T^\star / q_{max}$ (nếu symmetric).

## Bias correction
- Với lớp conv: $y = W * x + b$.  
- Sau lượng tử: $\hat{y} = \hat{W} * \hat{x} + \hat{b}$.  
- Ước lượng sai lệch:  
  $$
  \Delta_c = \mathbb{E}_{\mathcal{D}}[y_c - \hat{y}_c]
  $$
- Cập nhật bias:  
  $$
  \hat{b}_c \leftarrow \hat{b}_c + \Delta_c
  $$

## Trình tự
1. Train FP32, fuse Conv–BN–ReLU.  
2. Chèn observer histogram cho activations.  
3. Calibration (khoảng 1000 ảnh).  
4. Tìm ngưỡng tối ưu theo KL hoặc MSE.  
5. Convert sang INT8.  
6. Chạy bias correction bằng calibration set.  
7. Đánh giá lại.  


In [18]:
# [Cell 11] PTQ v2 (Histogram calibration + Bias Correction)
# - Load baseline FP32, dùng HistogramObserver cho activations + per-channel symmetric cho weights
# - Calibration (histogram), Convert → INT8, Bias correction (logit-level)
# - Lưu checkpoint ptqv2_MobileLiteC10.pth và so sánh với baseline (params, FLOPs, latency, acc, kích thước)

from __future__ import annotations
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from torch.ao.quantization import QConfig
from torch.ao.quantization.observer import HistogramObserver, PerChannelMinMaxObserver
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx


# ==== Kiến trúc (khớp phần train) ====
class ConvBNAct(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, g=1, act=True):
        super().__init__()
        p = (k - 1) // 2
        self.conv = nn.Conv2d(in_c, out_c, k, s, p, groups=g, bias=False)
        self.bn   = nn.BatchNorm2d(out_c)
        self.act  = act
    def forward(self, x):
        x = self.conv(x); x = self.bn(x)
        return F.relu6(x, inplace=True) if self.act else x

class InvertedResidual(nn.Module):
    def __init__(self, in_c, out_c, t=4, s=1, k=3):
        super().__init__()
        mid = int(round(in_c * t))
        self.use_res = (s == 1 and in_c == out_c)
        self.expand = ConvBNAct(in_c, mid, k=1, s=1, act=True) if t != 1 else nn.Identity()
        self.dw = ConvBNAct(mid, mid, k=k, s=s, g=mid, act=True)
        self.project = ConvBNAct(mid, out_c, k=1, s=1, act=False)
    def forward(self, x):
        y = self.expand(x) if not isinstance(self.expand, nn.Identity) else x
        y = self.dw(y); y = self.project(y)
        return x + y if self.use_res else y

class MobileLiteC10(nn.Module):
    def __init__(self, num_classes=10, alpha=0.75):
        super().__init__()
        def ch(x): return max(8, int(round(x * alpha)))
        self.stem = ConvBNAct(3, ch(32), k=3, s=1)
        layers = []
        layers += [InvertedResidual(ch(32), ch(24), t=4, s=1, k=3),
                   InvertedResidual(ch(24), ch(24), t=4, s=1, k=3)]
        layers += [InvertedResidual(ch(24), ch(32), t=4, s=2, k=3),
                   InvertedResidual(ch(32), ch(32), t=4, s=1, k=3),
                   InvertedResidual(ch(32), ch(32), t=4, s=1, k=3)]
        layers += [InvertedResidual(ch(32), ch(48), t=4, s=2, k=3),
                   InvertedResidual(ch(48), ch(48), t=4, s=1, k=3),
                   InvertedResidual(ch(48), ch(48), t=4, s=1, k=3)]
        layers += [InvertedResidual(ch(48), ch(64), t=4, s=1, k=3),
                   InvertedResidual(ch(64), ch(64), t=4, s=1, k=3),
                   InvertedResidual(ch(64), ch(64), t=4, s=1, k=3)]
        self.features = nn.Sequential(*layers)
        self.head_conv = ConvBNAct(ch(64), ch(160), k=1, s=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(ch(160), num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = self.head_conv(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

# ==== Utils ====
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_macs_flops(model: nn.Module, input_size=(1,3,32,32)):
    hooks = []; macs = 0
    def conv_hook(m, inp, out):
        nonlocal macs
        x = inp[0]
        B, Cin, Hin, Win = x.shape
        B, Cout, Hout, Wout = out.shape
        Kh, Kw = m.kernel_size
        groups = m.groups
        cin_per_group = Cin // groups
        macs += Cout * Hout * Wout * Kh * Kw * cin_per_group
    def linear_hook(m, inp, out):
        nonlocal macs
        macs += inp[0].shape[-1] * out.shape[-1]
    for m in model.modules():
        if isinstance(m, nn.Conv2d): hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear): hooks.append(m.register_forward_hook(linear_hook))
    model.eval()
    with torch.inference_mode():
        dummy = torch.randn(*input_size)
        _ = model(dummy)
    for h in hooks: h.remove()
    flops = 2 * macs
    return macs, flops

def eval_model_cpu(model: nn.Module, loader: DataLoader):
    model.eval()
    top1 = top5 = total = 0
    times = []
    with torch.inference_mode():
        for x, y in loader:
            if not times: _ = model(x)  # warmup
            t0 = time.time(); logits = model(x); t1 = time.time()
            times.append(t1 - t0)
            _, pred = logits.topk(5, 1, True, True)
            pred = pred.t()
            correct = pred.eq(y.view(1, -1))
            top1 += correct[:1].reshape(-1).float().sum().item()
            top5 += correct[:5].reshape(-1).float().sum().item()
            total += y.size(0)
    avg_batch = sum(times)/len(times)
    return {
        "top1": top1/total, "top5": top5/total,
        "avg_batch_time_s": avg_batch,
        "avg_per_sample_time_s": avg_batch/loader.batch_size
    }

def file_size_mb(path: Path) -> float:
    return Path(path).stat().st_size / (1024*1024)

# ==== Đường dẫn & backend ====
CHECKPOINT_DIR = Path("checkpoints"); CHECKPOINT_DIR.mkdir(exist_ok=True, parents=True)
FP32_CKPT = CHECKPOINT_DIR / "fp32_MobileLiteC10.pth"
PTQ2_CKPT = CHECKPOINT_DIR / "ptqv2_MobileLiteC10.pth"

supported = torch.backends.quantized.supported_engines
engine = "fbgemm" if "fbgemm" in supported else ("qnnpack" if "qnnpack" in supported else supported[0])
torch.backends.quantized.engine = engine

# ==== Datasets (calibration + test, KHÔNG augment) ====
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)
calib_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)])
test_tf  = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)])

calib_full = datasets.CIFAR10("./data", train=True,  transform=calib_tf, download=True)
test_set   = datasets.CIFAR10("./data", train=False, transform=test_tf,   download=True)
calib_idx = list(range(min(1024, len(calib_full))))
calib_loader = DataLoader(Subset(calib_full, calib_idx), batch_size=128, shuffle=False, num_workers=2, pin_memory=False)
test_loader  = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2, pin_memory=False)

# ==== 1) Load baseline FP32 ====
assert FP32_CKPT.exists(), f"Không tìm thấy baseline tại {FP32_CKPT}"
blob = torch.load(FP32_CKPT, map_location="cpu")
alpha = blob.get("cfg", {}).get("alpha", 0.75)
fp32_model = MobileLiteC10(num_classes=10, alpha=alpha).eval()
fp32_model.load_state_dict(blob["model"], strict=True)

# Baseline metrics
macs, flops = count_macs_flops(fp32_model, input_size=(1,3,32,32))
base_params = count_params(fp32_model)
base_metrics = eval_model_cpu(fp32_model, test_loader)
base_ckpt_mb = file_size_mb(FP32_CKPT)

# ==== 2) PTQ v2: Histogram qconfig + per-channel weight ====
act_obs = HistogramObserver.with_args(dtype=torch.quint8, qscheme=torch.per_tensor_affine, reduce_range=False)
w_obs   = PerChannelMinMaxObserver.with_args(dtype=torch.qint8, qscheme=torch.per_channel_symmetric, ch_axis=0)
qconfig = QConfig(activation=act_obs, weight=w_obs)
qconfig_dict = {"": qconfig}

# ---- prepare & calibrate ----
example_input = torch.randn(1,3,32,32)
prepared = prepare_fx(fp32_model, qconfig_dict, example_input)
with torch.inference_mode():
    for x, _ in calib_loader:
        _ = prepared(x)

# ---- convert (INT8 tạm) ----
int8_tmp = convert_fx(prepared).eval()

# ==== 3) Ước lượng Δ logits trên tập calibration ====
with torch.inference_mode():
    sum_fp32 = None; sum_int8 = None; n = 0
    for x, _ in calib_loader:
        logits_f = fp32_model(x)
        logits_q = int8_tmp(x)
        sum_fp32 = logits_f.sum(dim=0) if sum_fp32 is None else (sum_fp32 + logits_f.sum(dim=0))
        sum_int8 = logits_q.sum(dim=0) if sum_int8 is None else (sum_int8 + logits_q.sum(dim=0))
        n += x.size(0)
    delta = (sum_fp32 - sum_int8) / max(n, 1)  # vector size = num_classes

# ==== 4) Quay lại FP32, cộng Δ vào fc.bias rồi quantize lại ====
fp32_corr = MobileLiteC10(num_classes=10, alpha=alpha).eval()
fp32_corr.load_state_dict(blob["model"], strict=True)
with torch.no_grad():
    if fp32_corr.fc.bias is None:
        fp32_corr.fc.bias = nn.Parameter(delta.clone())
    else:
        fp32_corr.fc.bias.data.add_(delta)

prepared2 = prepare_fx(fp32_corr, qconfig_dict, example_input)
with torch.inference_mode():
    for x, _ in calib_loader:
        _ = prepared2(x)
int8_model = convert_fx(prepared2).eval()  # đây là mô hình INT8 sau bias correction

# ==== 5) Lưu checkpoint INT8 ====
torch.save({
    "state_dict": int8_model.state_dict(),
    "alpha": alpha,
    "engine": engine,
    "example_input_shape": (1,3,32,32)
}, PTQ2_CKPT)

# ==== 6) Đánh giá INT8 sau PTQ v2 ====
int8_metrics = eval_model_cpu(int8_model, test_loader)
ptq2_ckpt_mb = file_size_mb(PTQ2_CKPT)

# ==== 7) So sánh PTQ v2 vs Baseline ====
print("=== Baseline FP32 (CPU) ===")
print(f"Params: {base_params:,}")
print(f"MACs: {macs/1e6:.2f} M | FLOPs: {flops/1e6:.2f} MFLOPs")
print(f"Checkpoint size: {base_ckpt_mb:.2f} MB")
print(f"acc@1: {base_metrics['top1']*100:.2f}% | acc@5: {base_metrics['top5']*100:.2f}%")
print(f"Avg batch latency: {base_metrics['avg_batch_time_s']*1000:.3f} ms | "
      f"per-sample: {base_metrics['avg_per_sample_time_s']*1000:.3f} ms")

print("\n=== PTQ v2 INT8 (Histogram + Bias Correction, CPU) ===")
print(f"Params (logic tương đương).")
print(f"MACs: {macs/1e6:.2f} M | FLOPs: {flops/1e6:.2f} MFLOPs")
print(f"Checkpoint size: {ptq2_ckpt_mb:.2f} MB")
print(f"acc@1: {int8_metrics['top1']*100:.2f}% | acc@5: {int8_metrics['top5']*100:.2f}%")
print(f"Avg batch latency: {int8_metrics['avg_batch_time_s']*1000:.3f} ms | "
      f"per-sample: {int8_metrics['avg_per_sample_time_s']*1000:.3f} ms")

print("\n=== Δ (FP32 → PTQ v2) ===")
print(f"acc@1 Δ: {(int8_metrics['top1']-base_metrics['top1'])*100:+.2f} pp | "
      f"acc@5 Δ: {(int8_metrics['top5']-base_metrics['top5'])*100:+.2f} pp")
print(f"Latency per-sample Δ: {(base_metrics['avg_per_sample_time_s']-int8_metrics['avg_per_sample_time_s'])*1000:+.3f} ms")
print(f"Checkpoint Δ size: {base_ckpt_mb - ptq2_ckpt_mb:+.2f} MB")

/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/quantize_fx.py:146: FutureWarning: Passing a QConfig dictionary to prepare is deprecated and will not be supported in a future version. Please pass in a QConfigMapping instead.
  prepared = prepare(
/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/quantize_fx.py:146: FutureWarning: Passing a QConfig dictionary to prepare is deprecated and will not be supported in a future version. Please pass in a QConfigMapping instead.
  prepared = prepare(


=== Baseline FP32 (CPU) ===
Params: 765,898
MACs: 126.63 M | FLOPs: 253.25 MFLOPs
Checkpoint size: 3.06 MB
acc@1: 90.34% | acc@5: 99.73%
Avg batch latency: 1862.060 ms | per-sample: 14.547 ms

=== PTQ v2 INT8 (Histogram + Bias Correction, CPU) ===
Params (logic tương đương).
MACs: 126.63 M | FLOPs: 253.25 MFLOPs
Checkpoint size: 0.95 MB
acc@1: 88.85% | acc@5: 99.64%
Avg batch latency: 391.095 ms | per-sample: 3.055 ms

=== Δ (FP32 → PTQ v2) ===
acc@1 Δ: -1.49 pp | acc@5 Δ: -0.09 pp
Latency per-sample Δ: +11.492 ms
Checkpoint Δ size: +2.11 MB


In [19]:
# [Cell 12] PTQ v2 — FINAL EVAL CELL (độc lập)
# - Tự xây graph quant theo backend, load state_dict INT8, đo params (logic), FLOPs, latency, acc
# - Không phụ thuộc bất kỳ biến từ cell trước

import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
from torch.ao.quantization.observer import HistogramObserver, PerChannelMinMaxObserver

# ==== Kiến trúc ====
class ConvBNAct(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, g=1, act=True):
        super().__init__()
        p = (k - 1) // 2
        self.conv = nn.Conv2d(in_c, out_c, k, s, p, groups=g, bias=False)
        self.bn   = nn.BatchNorm2d(out_c)
        self.act  = act
    def forward(self, x):
        x = self.conv(x); x = self.bn(x)
        return F.relu6(x, inplace=True) if self.act else x

class InvertedResidual(nn.Module):
    def __init__(self, in_c, out_c, t=4, s=1, k=3):
        super().__init__()
        mid = int(round(in_c * t))
        self.use_res = (s == 1 and in_c == out_c)
        self.expand = ConvBNAct(in_c, mid, k=1, s=1, act=True) if t != 1 else nn.Identity()
        self.dw = ConvBNAct(mid, mid, k=k, s=s, g=mid, act=True)
        self.project = ConvBNAct(mid, out_c, k=1, s=1, act=False)
    def forward(self, x):
        y = self.expand(x) if not isinstance(self.expand, nn.Identity) else x
        y = self.dw(y); y = self.project(y)
        return x + y if self.use_res else y

class MobileLiteC10(nn.Module):
    def __init__(self, num_classes=10, alpha=0.75):
        super().__init__()
        def ch(x): return max(8, int(round(x * alpha)))
        self.stem = ConvBNAct(3, ch(32), k=3, s=1)
        layers = []
        layers += [InvertedResidual(ch(32), ch(24), t=4, s=1, k=3),
                   InvertedResidual(ch(24), ch(24), t=4, s=1, k=3)]
        layers += [InvertedResidual(ch(24), ch(32), t=4, s=2, k=3),
                   InvertedResidual(ch(32), ch(32), t=4, s=1, k=3),
                   InvertedResidual(ch(32), ch(32), t=4, s=1, k=3)]
        layers += [InvertedResidual(ch(32), ch(48), t=4, s=2, k=3),
                   InvertedResidual(ch(48), ch(48), t=4, s=1, k=3),
                   InvertedResidual(ch(48), ch(48), t=4, s=1, k=3)]
        layers += [InvertedResidual(ch(48), ch(64), t=4, s=1, k=3),
                   InvertedResidual(ch(64), ch(64), t=4, s=1, k=3),
                   InvertedResidual(ch(64), ch(64), t=4, s=1, k=3)]
        self.features = nn.Sequential(*layers)
        self.head_conv = ConvBNAct(ch(64), ch(160), k=1, s=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(ch(160), num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = self.head_conv(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_macs_flops_any(model: nn.Module, input_size=(1,3,32,32)):
    import torch.nn.quantized as nnq
    hooks = []
    macs = 0
    def conv_hook(m, inp, out):
        nonlocal macs
        x = inp[0]
        B, Cin, Hin, Win = x.shape
        B, Cout, Hout, Wout = out.shape
        Kh, Kw = m.kernel_size
        groups = getattr(m, "groups", 1)
        cin_per_group = Cin // groups
        macs += Cout * Hout * Wout * Kh * Kw * cin_per_group
    def linear_hook(m, inp, out):
        nonlocal macs
        macs += inp[0].shape[-1] * out.shape[-1]
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nnq.Conv2d)):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, (nn.Linear, nnq.Linear)):
            hooks.append(m.register_forward_hook(linear_hook))
    model.eval()
    with torch.inference_mode():
        dummy = torch.randn(*input_size)
        _ = model(dummy)
    for h in hooks: h.remove()
    flops = 2 * macs
    return macs, flops

@torch.inference_mode()
def evaluate_cpu(model: nn.Module, loader: DataLoader):
    top1 = top5 = total = 0
    times = []
    for x, y in loader:
        if not times:
            _ = model(x)
        t0 = time.time()
        logits = model(x)
        t1 = time.time()
        times.append(t1 - t0)
        _, pred = logits.topk(5, 1, True, True)
        pred = pred.t()
        correct = pred.eq(y.view(1, -1))
        top1 += correct[:1].reshape(-1).float().sum().item()
        top5 += correct[:5].reshape(-1).float().sum().item()
        total += y.size(0)
    avg_batch = sum(times)/len(times)
    return {
        "top1": top1/total, "top5": top5/total,
        "avg_batch_time_s": avg_batch,
        "avg_per_sample_time_s": avg_batch/loader.batch_size
    }

# ==== Load checkpoint INT8 ====
CKPT = Path("checkpoints/ptqv2_MobileLiteC10.pth")
assert CKPT.exists(), f"Không tìm thấy checkpoint: {CKPT}"
blob = torch.load(CKPT, map_location="cpu")
alpha = blob.get("alpha", 0.75)
engine = blob.get("engine", "fbgemm")
example_shape = blob.get("example_input_shape", (1,3,32,32))
torch.backends.quantized.engine = engine

# ==== Build quant graph & load state ====
fp32_model = MobileLiteC10(alpha=alpha).eval()
act_obs = HistogramObserver.with_args(dtype=torch.quint8, qscheme=torch.per_tensor_affine, reduce_range=False)
w_obs   = PerChannelMinMaxObserver.with_args(dtype=torch.qint8, qscheme=torch.per_channel_symmetric, ch_axis=0)
qconfig = QConfig(activation=act_obs, weight=w_obs)
qconfig_dict = {"": qconfig}
prepared = prepare_fx(fp32_model, qconfig_dict, torch.randn(*example_shape))
int8_model = convert_fx(prepared).eval()
int8_model.load_state_dict(blob["state_dict"], strict=True)

# ==== DataLoader test ====
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)])
test_set = datasets.CIFAR10("./data", train=False, transform=test_tf, download=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2, pin_memory=False)

# ==== Metrics ====
params_logic = count_params(fp32_model)
macs, flops = count_macs_flops_any(int8_model, input_size=example_shape)
metrics = evaluate_cpu(int8_model, test_loader)

print(f"Engine: {engine}")
print(f"Params (logic FP32 graph): {params_logic:,}")
print(f"MACs: {macs/1e6:.2f} M | FLOPs: {flops/1e6:.2f} MFLOPs")
print(f"acc@1: {metrics['top1']*100:.2f}% | acc@5: {metrics['top5']*100:.2f}%")
print(f"Avg batch latency: {metrics['avg_batch_time_s']*1000:.3f} ms | "
      f"Avg per-sample: {metrics['avg_per_sample_time_s']*1000:.3f} ms")


/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/quantize_fx.py:146: FutureWarning: Passing a QConfig dictionary to prepare is deprecated and will not be supported in a future version. Please pass in a QConfigMapping instead.
  prepared = prepare(
/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/observer.py:1318: UserWarning: must run observer before calling calculate_qparams.                                    Returning default scale and zero point 
  warnings.warn(


Engine: fbgemm
Params (logic FP32 graph): 765,898
MACs: 126.63 M | FLOPs: 253.25 MFLOPs
acc@1: 88.85% | acc@5: 99.64%
Avg batch latency: 373.491 ms | Avg per-sample: 2.918 ms
